# RSICD Per-Class Patch Vocabulary Generation

### Goal
Build a caption-grounded vocabulary for patch-level classification, structured as:
- **A global vocabulary** — one deduplicated list of visual terms across all 31 RSICD scene classes.
- **Per-class subsets** — each scene class (airport, farmland, port, etc.) mapped to only the
  subset of the global vocabulary that's actually plausible for that scene type.

This constrains patch-level zero-shot classification to a small, relevant candidate list per
image (based on its scene class), instead of the full vocabulary — directly addressing the
CLIP zero-shot "hub"/class-imbalance bias observed when classifying patches against the full,
unconstrained vocabulary.

**Pipeline:**
1. Setup — install dependencies, load spaCy
2. Load RSICD captions JSON + scene-class label files
3. Group and deduplicate captions per scene class
4. Extract candidate nouns/noun-phrases per class using spaCy (batched)
5. Write raw per-class term files (for manual review/curation)
6. Cross-class term overlap diagnostic
7. Load the manually curated vocabulary
8. Build the global vocabulary (deduplicated, indexed)
9. Build term → classes and class → vocabulary mappings
10. Build the final class → vocabulary-with-indices mapping (ready for model use)


## 1. Setup

In [1]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 57.2 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import json
import os
import time
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import spacy

nlp = spacy.load("en_core_web_sm")

In [3]:
# Dataset paths -- update these to match your mounted dataset location
BASE_DIR = "/kaggle/input/datasets/vedantbarhate007/rsicd-eric/RSICD_optimal-master"
CLASSES_DIR = os.path.join(BASE_DIR, "txtclasses_rsicd")
JSON_PATH = os.path.join(BASE_DIR, "dataset_rsicd.json")

OUTPUT_DIR = "/kaggle/working/class_vocab_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

assert os.path.isdir(CLASSES_DIR), f"Classes dir not found: {CLASSES_DIR}"
assert os.path.isfile(JSON_PATH), f"JSON not found: {JSON_PATH}"
print("Paths OK")

Paths OK


## 2. Load RSICD Captions JSON + Scene-Class Label Files

`dataset_rsicd.json` holds every image's captions (`sentences[].raw`). The `txtclasses_rsicd`
folder holds one `.txt` file per scene class, each listing the image filenames belonging to
that class. We use the latter to build a `filename -> class_index` lookup.

In [4]:
with open(JSON_PATH) as f:
    data = json.load(f)

print("Top-level keys:", data.keys())
print("Number of images:", len(data["images"]))
print("Sample entry:")
print(data["images"][0])

Top-level keys: dict_keys(['images', 'dataset'])
Number of images: 10921
Sample entry:
{'filename': 'airport_1.jpg', 'imgid': 0, 'sentences': [{'tokens': ['many', 'planes', 'are', 'parked', 'next', 'to', 'a', 'long', 'building', 'in', 'an', 'airport'], 'raw': 'many planes are parked next to a long building in an airport .', 'imgid': 0, 'sentid': 0}, {'tokens': ['many', 'planes', 'are', 'parked', 'next', 'to', 'a', 'long', 'building', 'in', 'an', 'airport'], 'raw': 'many planes are parked next to a long building in an airport .', 'imgid': 0, 'sentid': 1}, {'tokens': ['many', 'planes', 'are', 'parked', 'next', 'to', 'a', 'long', 'building', 'in', 'an', 'airport'], 'raw': 'many planes are parked next to a long building in an airport .', 'imgid': 0, 'sentid': 2}, {'tokens': ['many', 'planes', 'are', 'parked', 'next', 'to', 'a', 'long', 'building', 'in', 'an', 'airport'], 'raw': 'many planes are parked next to a long building in an airport .', 'imgid': 0, 'sentid': 3}, {'tokens': ['many', '

In [5]:
files = sorted(f for f in os.listdir(CLASSES_DIR) if f.endswith(".txt"))
classes = [os.path.splitext(f)[0].lower() for f in files]

filename_to_class = {}  # filename -> class index

for class_idx, fname in enumerate(files):
    filepath = os.path.join(CLASSES_DIR, fname)
    with open(filepath) as f:
        image_names = [line.strip() for line in f if line.strip()]
    for img_name in image_names:
        filename_to_class[img_name] = class_idx

print(f"Classes ({len(classes)}): {classes}")
print(f"Total mapped images: {len(filename_to_class)}")

Classes (31): ['airport', 'bareland', 'baseballfield', 'beach', 'bridge', 'center', 'church', 'commercial', 'denseresidential', 'desert', 'farmland', 'forest', 'industrial', 'meadow', 'mediumresidential', 'mountain', 'park', 'parking', 'playground', 'pond', 'port', 'railwaystation', 'resort', 'river', 'school', 'sparseresidential', 'square', 'stadium', 'storagetanks', 'viaduct', 'playfields']
Total mapped images: 10921


## 3. Group and Deduplicate Captions per Scene Class

RSICD contains a large proportion of exact-duplicate captions (~66% in our data). Deduplicating
per class (case/whitespace-normalized) before noun extraction avoids inflating term frequency
with repeated phrasing rather than genuine content diversity.

In [6]:
captions_by_class = defaultdict(set)  # class_idx -> set of unique, normalized captions

unmatched = 0
total_raw = 0
for img_entry in data["images"]:
    fname = img_entry["filename"]
    if fname not in filename_to_class:
        unmatched += 1
        continue
    class_idx = filename_to_class[fname]
    for sent in img_entry["sentences"]:
        raw = sent["raw"].strip().lower()
        captions_by_class[class_idx].add(raw)
        total_raw += 1

print(f"Unmatched images (no class label found): {unmatched}")
print(f"Total raw captions seen: {total_raw}")

total_unique = 0
for idx in range(len(classes)):
    n_unique = len(captions_by_class[idx])
    total_unique += n_unique
    print(f"{classes[idx]}: {n_unique} unique captions")

print(f"\nTotal unique captions across all classes: {total_unique}")
print(f"Duplicate rate: {100 * (1 - total_unique/total_raw):.1f}%")

Unmatched images (no class label found): 0
Total raw captions seen: 54605
airport: 782 unique captions
bareland: 391 unique captions
baseballfield: 577 unique captions
beach: 567 unique captions
bridge: 1038 unique captions
center: 486 unique captions
church: 407 unique captions
commercial: 435 unique captions
denseresidential: 402 unique captions
desert: 389 unique captions
farmland: 449 unique captions
forest: 275 unique captions
industrial: 546 unique captions
meadow: 355 unique captions
mediumresidential: 318 unique captions
mountain: 380 unique captions
park: 614 unique captions
parking: 536 unique captions
playground: 586 unique captions
pond: 712 unique captions
port: 606 unique captions
railwaystation: 320 unique captions
resort: 515 unique captions
river: 607 unique captions
school: 462 unique captions
sparseresidential: 453 unique captions
square: 652 unique captions
stadium: 511 unique captions
storagetanks: 816 unique captions
viaduct: 654 unique captions
playfields: 2507 u

## 4. Extract Candidate Nouns / Noun-Phrases per Class (spaCy, Batched)

For each scene class, run all its unique captions through spaCy's pipeline (batched via
`nlp.pipe()` for speed) and collect individual noun lemmas plus short noun chunks
(1–3 words, e.g. "parking lot", "green tree") as candidate vocabulary terms.

In [7]:
def extract_nouns_from_doc(doc):
    """
    Extract singular-noun lemmas and noun chunks from an already-parsed spaCy doc.
    """
    terms = []

    for token in doc:
        if token.pos_ == "NOUN" and not token.is_stop:
            terms.append(token.lemma_.lower())

    for chunk in doc.noun_chunks:
        cleaned = " ".join(
            t.lemma_.lower() for t in chunk if not t.is_stop and t.pos_ in ("NOUN", "ADJ", "PROPN")
        ).strip()
        if cleaned and 1 <= len(cleaned.split()) <= 3:
            terms.append(cleaned)

    return terms

In [8]:
noun_counts_by_class = {}
start = time.time()

for class_idx in range(len(classes)):
    captions = list(captions_by_class[class_idx])
    counter = Counter()

    for doc in nlp.pipe(captions, batch_size=256, n_process=1):
        terms = extract_nouns_from_doc(doc)
        counter.update(terms)

    noun_counts_by_class[class_idx] = counter
    print(f"{classes[class_idx]}: {len(captions)} captions processed, {len(counter)} unique terms")

elapsed = time.time() - start
print(f"\nTotal time: {elapsed:.1f}s")

airport: 782 captions processed, 428 unique terms
bareland: 391 captions processed, 343 unique terms
baseballfield: 577 captions processed, 352 unique terms
beach: 567 captions processed, 351 unique terms
bridge: 1038 captions processed, 571 unique terms
center: 486 captions processed, 427 unique terms
church: 407 captions processed, 314 unique terms
commercial: 435 captions processed, 350 unique terms
denseresidential: 402 captions processed, 283 unique terms
desert: 389 captions processed, 373 unique terms
farmland: 449 captions processed, 385 unique terms
forest: 275 captions processed, 242 unique terms
industrial: 546 captions processed, 376 unique terms
meadow: 355 captions processed, 291 unique terms
mediumresidential: 318 captions processed, 237 unique terms
mountain: 380 captions processed, 229 unique terms
park: 614 captions processed, 392 unique terms
parking: 536 captions processed, 292 unique terms
playground: 586 captions processed, 313 unique terms
pond: 712 captions proc

In [9]:
# Spot-check a few classes' top terms by frequency
for idx in [0, 5, 10, 20]:
    print(f"\n=== {classes[idx]} ===")
    for term, count in noun_counts_by_class[idx].most_common(10):
        print(f"  {term}: {count}")


=== airport ===
  plane: 933
  airport: 917
  building: 478
  runway: 404
  terminal: 144
  parking: 114
  lot: 102
  apron: 83
  parking lot: 67
  road: 64

=== center ===
  building: 500
  center: 298
  tree: 224
  road: 219
  green tree: 137
  lot: 77
  car: 72
  circle: 66
  parking: 52
  oval: 51

=== farmland ===
  farmland: 444
  piece: 207
  road: 125
  field: 82
  tree: 72
  farm: 70
  land: 62
  block: 53
  green farmland: 49
  lot: 32

=== port ===
  port: 766
  boat: 565
  building: 297
  ship: 183
  tree: 102
  road: 79
  harbour: 69
  lot: 66
  bank: 63
  green tree: 62


## 5. Write Raw Per-Class Term Files (for Manual Review)

These are the **unfiltered** extracted terms, one plain word/phrase per line (no frequencies),
written out per class for manual curation — pruning generic/noisy terms (e.g. "piece", "lot",
"block") down to a clean, visually meaningful vocabulary per class.

In [10]:
for class_idx in range(len(classes)):
    class_name = classes[class_idx]
    terms = sorted(noun_counts_by_class[class_idx].keys())

    output_path = os.path.join(OUTPUT_DIR, f"{class_name}.txt")
    with open(output_path, "w") as f:
        for term in terms:
            f.write(term + "\n")

    print(f"{class_name}: {len(terms)} unique terms -> {output_path}")

print(f"\nAll files written to: {OUTPUT_DIR}")

airport: 428 unique terms -> /kaggle/working/class_vocab_outputs/airport.txt
bareland: 343 unique terms -> /kaggle/working/class_vocab_outputs/bareland.txt
baseballfield: 352 unique terms -> /kaggle/working/class_vocab_outputs/baseballfield.txt
beach: 351 unique terms -> /kaggle/working/class_vocab_outputs/beach.txt
bridge: 571 unique terms -> /kaggle/working/class_vocab_outputs/bridge.txt
center: 427 unique terms -> /kaggle/working/class_vocab_outputs/center.txt
church: 314 unique terms -> /kaggle/working/class_vocab_outputs/church.txt
commercial: 350 unique terms -> /kaggle/working/class_vocab_outputs/commercial.txt
denseresidential: 283 unique terms -> /kaggle/working/class_vocab_outputs/denseresidential.txt
desert: 373 unique terms -> /kaggle/working/class_vocab_outputs/desert.txt
farmland: 385 unique terms -> /kaggle/working/class_vocab_outputs/farmland.txt
forest: 242 unique terms -> /kaggle/working/class_vocab_outputs/forest.txt
industrial: 376 unique terms -> /kaggle/working/cl

In [11]:
# Zip for download from Kaggle's output panel
shutil.make_archive("/kaggle/working/class_vocab_outputs", "zip", OUTPUT_DIR)
print("Zipped to /kaggle/working/class_vocab_outputs.zip")

Zipped to /kaggle/working/class_vocab_outputs.zip


## 6. Cross-Class Term Overlap Diagnostic

Before manual pruning, it helps to see which raw terms are common across many classes (likely
generic/universal visual elements, e.g. "tree", "road", "building") versus which are unique to
a single class (likely genuinely distinctive, e.g. "apron" for airport) — this guides the
manual curation pass.

In [12]:
term_to_classes_raw = defaultdict(set)
for class_idx in range(len(classes)):
    for term in noun_counts_by_class[class_idx].keys():
        term_to_classes_raw[term].add(classes[class_idx])

sorted_terms = sorted(term_to_classes_raw.items(), key=lambda x: -len(x[1]))

print("Most cross-class-common terms (appear in many classes):")
for term, cls_set in sorted_terms[:20]:
    print(f"  {term}: appears in {len(cls_set)} classes")

unique_terms = [t for t, c in term_to_classes_raw.items() if len(c) == 1]
print(f"\nTotal terms unique to a single class: {len(unique_terms)}")
print(f"Sample: {unique_terms[:20]}")

Most cross-class-common terms (appear in many classes):
  ground: appears in 31 classes
  road: appears in 31 classes
  lot: appears in 31 classes
  land: appears in 31 classes
  area: appears in 31 classes
  plant: appears in 31 classes
  green plant: appears in 31 classes
  river: appears in 31 classes
  piece: appears in 30 classes
  tree: appears in 30 classes
  grass: appears in 30 classes
  green tree: appears in 30 classes
  number: appears in 29 classes
  large number: appears in 29 classes
  building: appears in 29 classes
  line: appears in 29 classes
  row: appears in 29 classes
  middle: appears in 29 classes
  house: appears in 29 classes
  roof: appears in 29 classes

Total terms unique to a single class: 4664
Sample: ['isoscele', 'trapezoid', 'apron', 'aeroplane', 'brown grass', 'large flat', 'terminal', 'rectangular terminal building', 'termial', 'large apron', 'triangular blue', 'parallel runway', 'passenger termial building', 'large airport', 'pink gray building', 'ta

> **Manual step (outside this notebook):** review each class's raw term file, prune to a
> clean, visually meaningful list (~15-20 terms per class), and save the curated `.txt` files
> to a new folder — referenced below as `VOCAB_DIR`. Watch specifically for near-synonym
> clusters *within* the same class (e.g. "tank" / "storage tank" / "oil tank" / "cylinder tank"
> all competing for the same visual signal) — collapsing these to one representative term
> avoids reintroducing the same hub/bias problem this vocabulary restructuring is meant to fix.

## 7. Load the Manually Curated Vocabulary

In [13]:
VOCAB_DIR = Path("/kaggle/input/datasets/vedantbarhate007/eric-raster-subclasses/picked_class_vocab")

assert VOCAB_DIR.is_dir(), f"Curated vocab dir not found: {VOCAB_DIR}"
print(f"Found {len(list(VOCAB_DIR.glob('*.txt')))} curated class files")

Found 31 curated class files


## 8. Build the Global Vocabulary

Deduplicate terms across all 31 curated class files into a single, indexed global vocabulary —
this is the full candidate list whose text embeddings get computed once and reused across
every scene class's subset.

In [14]:
global_vocab_terms = set()

for txt_file in VOCAB_DIR.glob("*.txt"):
    with txt_file.open("r", encoding="utf-8") as f:
        for line in f:
            word = line.strip().lower()
            if word:
                global_vocab_terms.add(word)

global_vocab_terms = sorted(global_vocab_terms)

print(f"Total unique terms: {len(global_vocab_terms)}")
print(global_vocab_terms)

Total unique terms: 210
['aeroplane', 'airport', 'apartment', 'apron', 'asphalt', 'awning', 'badminton court', 'ball', 'bank', 'bare ground', 'bare land', 'bareland', 'barren', 'base', 'baseball field', 'basketball court', 'beach', 'beach chair', 'bleacher', 'boat', 'bridge', 'bridge deck', 'bridge pier', 'building', 'bus', 'bush', 'business district', 'cable', 'car', 'cargo', 'cattle', 'center', 'church', 'cloud', 'coast', 'coastline', 'commercial building', 'container', 'cottage', 'courtyard', 'crack', 'crane', 'crop', 'cropland', 'cross', 'cruciform', 'dam', 'desert', 'dirt', 'dirt road', 'dock', 'dome', 'downtown', 'dry river', 'factory', 'farm track', 'farmland', 'fence', 'ferris wheel', 'field', 'fish', 'flower bed', 'football field', 'footpath', 'forest', 'fountain', 'garden', 'glass curtainwall', 'goal', 'grandstand', 'grass', 'gully', 'gym', 'gymnasium', 'harbor basin', 'highway', 'hill', 'hotel', 'house', 'hummock', 'industrial park', 'infield', 'intersection', 'island', 'jet

In [15]:
GLOBAL_VOCAB_FILE = Path("/kaggle/working/") / "global_vocab.json"

indexed_vocab = [
    {"index": i, "term": word}
    for i, word in enumerate(global_vocab_terms)
]

with GLOBAL_VOCAB_FILE.open("w", encoding="utf-8") as f:
    json.dump(indexed_vocab, f, indent=4, ensure_ascii=False)

print(f"Saved {len(indexed_vocab)} terms to {GLOBAL_VOCAB_FILE}")

Saved 210 terms to /kaggle/working/global_vocab.json


## 9. Build `term_to_classes.json` and `class_to_vocab.json`

- `term_to_classes.json` — reverse lookup: for each global vocabulary term, which classes use it
  (and how many). Useful for spotting overly generic terms that appear in nearly every class.
- `class_to_vocab.json` — the curated per-class term lists, as plain strings.

In [16]:
OUTPUT_FILE2 = Path("/kaggle/working/") / "term_to_classes.json"

with GLOBAL_VOCAB_FILE.open("r", encoding="utf-8") as f:
    global_vocab = json.load(f)

term_to_index = {item["term"]: item["index"] for item in global_vocab}

term_to_classes = {}

for txt_file in VOCAB_DIR.glob("*.txt"):
    class_name = txt_file.stem.lower()
    with txt_file.open("r", encoding="utf-8") as f:
        for line in f:
            term = line.strip().lower()
            if not term:
                continue
            term_to_classes.setdefault(term, set()).add(class_name)

output = []
for term, cls_set in term_to_classes.items():
    output.append({
        "index": term_to_index[term],
        "term": term,
        "class_count": len(cls_set),
        "classes": sorted(cls_set)
    })

output.sort(key=lambda x: x["index"])

with OUTPUT_FILE2.open("w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"Saved {len(output)} terms to {OUTPUT_FILE2}")

Saved 210 terms to /kaggle/working/term_to_classes.json


In [17]:
CLASS_TO_VOCAB_FILE = Path("/kaggle/working/") / "class_to_vocab.json"

class_to_vocab = {}

for txt_file in VOCAB_DIR.glob("*.txt"):
    class_name = txt_file.stem.lower()
    with txt_file.open("r", encoding="utf-8") as f:
        vocab = [line.strip().lower() for line in f if line.strip()]
    class_to_vocab[class_name] = vocab

class_to_vocab = dict(sorted(class_to_vocab.items()))

with CLASS_TO_VOCAB_FILE.open("w", encoding="utf-8") as f:
    json.dump(class_to_vocab, f, indent=4, ensure_ascii=False)

print(f"Saved {len(class_to_vocab)} classes to {CLASS_TO_VOCAB_FILE}")

Saved 31 classes to /kaggle/working/class_to_vocab.json


## 10. Build the Final `class_to_vocab_with_indices.json`

This is the mapping actually needed for patch classification: each scene class's curated terms,
resolved to their index in the global vocabulary. This lets you pre-compute the full global
vocabulary's text embeddings once, then select/mask just the relevant subset per image based on
its scene class.

In [18]:
OUTPUT_FILE = Path("/kaggle/working/") / "class_to_vocab_with_indices.json"

with CLASS_TO_VOCAB_FILE.open("r", encoding="utf-8") as f:
    class_to_vocab = json.load(f)

class_to_vocab_with_indices = {}

for class_name, vocab_list in class_to_vocab.items():
    indexed_vocab = []
    for term in vocab_list:
        term = term.strip().lower()
        if term not in term_to_index:
            print(f"Warning: '{term}' in class '{class_name}' not found in global vocabulary")
            continue
        indexed_vocab.append({"index": term_to_index[term], "term": term})
    class_to_vocab_with_indices[class_name] = indexed_vocab

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    json.dump(class_to_vocab_with_indices, f, indent=4, ensure_ascii=False)

print(f"Saved {len(class_to_vocab_with_indices)} classes to {OUTPUT_FILE}")

Saved 31 classes to /kaggle/working/class_to_vocab_with_indices.json


### Output Summary

| File | Contents |
|---|---|
| `global_vocab.json` | Deduplicated, indexed global vocabulary (all classes combined) |
| `class_to_vocab.json` | Curated per-class term lists (plain strings) |
| `term_to_classes.json` | Reverse lookup: term → which classes use it |
| `class_to_vocab_with_indices.json` | **Final, model-ready** mapping: class → list of `{index, term}` into the global vocabulary |

**Next step:** use `class_to_vocab_with_indices.json` to restrict each image's patch classification
candidates to only its scene class's subset — encode the full `global_vocab.json` once with
RemoteCLIP's text encoder, then select the relevant rows per image at inference time.
